# Local LoRA Fine-Tuning Run

This notebook performs the local training step: load examples, tokenize, mask prompt tokens, attach a LoRA adapter, run a small optimization loop, and save adapter weights. [SmolLM2-135M-Instruct](https://huggingface.co/HuggingFaceTB/SmolLM2-135M-Instruct) is compact enough for a laptop exercise while still producing interpretable before-and-after behavior.

The twelve supplied examples keep every stage of supervised adapter training visible and runnable. They demonstrate the workflow and produce an inspectable adapter, but they are intentionally too small for a production support model.


## Learning Objectives

By the end of this notebook, you should be able to:

- Prepare native chat examples with response-only loss and correctly masked padding.
- Train a real local LoRA adapter and inspect its parameters and loss history.
- Save adapter artifacts and run a generation smoke test without assuming generalization.


## Training Pipeline

Each section below implements one step of this pipeline.

```mermaid
flowchart LR
    A["Instruction JSONL"] --> B["Prompt + response"]
    B --> C["Tokenize and mask prompt"]
    C --> D["Base model + LoRA"]
    D --> E["Response-only loss"]
    E --> F["Saved adapter"]
```


In [ ]:
import hashlib
import json
import os
import random
from pathlib import Path

os.environ["TOKENIZERS_PARALLELISM"] = "false"

import numpy as np
import torch
from peft import LoraConfig, TaskType, get_peft_model
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "HuggingFaceTB/SmolLM2-135M-Instruct"
MODEL_REVISION = "12fd25f77366fa6b3b4b768ec3050bf629380bac"
OUTPUT_DIR = Path("outputs") / "lora-support-adapter"
DEVICE = torch.device("cpu")
SEED = 7

In [ ]:
def set_seed(seed: int) -> None:
    # Keep the small training run reproducible across reruns.
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


set_seed(SEED)

## Load and Serialize Examples

The model sees token sequences, not Python dictionaries. We use its [native chat template](https://huggingface.co/docs/transformers/chat_templating) to render the roles introduced in notebook 01. The prompt ends at the assistant-turn marker; the target contains the answer and its end-of-turn token. Training and inference use the same format.


In [ ]:
# Keep the exact input bytes for the adapter's training-data fingerprint.
training_bytes = Path("data/support_instruction_examples.jsonl").read_bytes()
training_records = [json.loads(line) for line in training_bytes.splitlines()]
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, revision=MODEL_REVISION)
tokenizer.pad_token = tokenizer.eos_token
SYSTEM_MESSAGE = "You are a concise support assistant."


def build_inference_prompt(instruction: str) -> str:
    # The same roles and assistant-turn marker are used during evaluation.
    messages = [
        {"role": "system", "content": SYSTEM_MESSAGE},
        {"role": "user", "content": instruction},
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )


def format_training_pair(record: dict[str, object]) -> tuple[str, str]:
    response = str(record["response"])
    if not response.strip():
        raise ValueError("Training answers must not be empty.")
    prompt = build_inference_prompt(str(record["instruction"]))
    messages = [
        {"role": "system", "content": SYSTEM_MESSAGE},
        {"role": "user", "content": str(record["instruction"])},
        {"role": "assistant", "content": response},
    ]
    full_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )
    if not full_text.startswith(prompt):
        raise ValueError(
            "The training template does not preserve the inference prefix."
        )
    # Keep the template's end-of-turn token; do not append a second EOS.
    return prompt, full_text[len(prompt) :]


training_pairs = [format_training_pair(record) for record in training_records]
training_pairs[0]

## Tokenize for Causal Language Modeling

A causal language model predicts the next token. The input contains both prompt and answer so the model has context, but the labels use `-100` for prompt and padding positions. [PyTorch cross-entropy loss](https://pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html) ignores that value, so optimization focuses on answer tokens rather than teaching the model to reproduce the user instruction.


In [ ]:
MAX_LENGTH = 128

The dataset class performs four jobs for each example: combine prompt and response, pad to a stable length, reject overlong examples, and mask labels that should not contribute to loss. Keeping those decisions together makes it easier to inspect one complete training item.


In [ ]:
class InstructionDataset(Dataset):
    # Dataset returning tokenized tensors and response-only labels.
    def __init__(self, pairs: list[tuple[str, str]]) -> None:
        self.pairs = pairs

    def __len__(self) -> int:
        return len(self.pairs)

    def __getitem__(self, index: int) -> dict[str, torch.Tensor]:
        prompt, response = self.pairs[index]
        # Preserve the native template and verify the response-loss boundary.
        prompt_ids = tokenizer.encode(prompt, add_special_tokens=False)
        token_ids = tokenizer.encode(prompt + response, add_special_tokens=False)
        if token_ids[: len(prompt_ids)] != prompt_ids:
            raise ValueError("Tokenization changed the prompt/answer boundary.")
        if (
            not response.strip()
            or len(token_ids) <= len(prompt_ids)
            or len(token_ids) > MAX_LENGTH
        ):
            raise ValueError(
                "Empty or overlong training answer; review the example or increase MAX_LENGTH."
            )
        padding = MAX_LENGTH - len(token_ids)
        input_ids = torch.tensor(token_ids + [tokenizer.pad_token_id] * padding)
        attention_mask = torch.tensor([1] * len(token_ids) + [0] * padding)
        labels = input_ids.clone()

        # Mask prompt and padding, but keep the genuine end-of-answer token.
        labels[: len(prompt_ids)] = -100
        labels[attention_mask == 0] = -100
        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels,
        }

In [ ]:
train_dataset = InstructionDataset(training_pairs)
# Validate every example before starting optimization, not just the first batch.
for index in range(len(train_dataset)):
    train_dataset[index]
train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True)
first_batch = next(iter(train_loader))
{key: value.shape for key, value in first_batch.items()}

### Exercise: Count Supervised Tokens

Count the non-`-100` labels in each item of the first batch.


In [ ]:
# @TODO: Count labels that contribute to the loss for each batch item.


<details>
<summary>Show Answer</summary>

```python
supervised_tokens_per_item = (first_batch["labels"] != -100).sum(dim=1)
supervised_tokens_per_item.tolist()
```

</details>


### Exercise: Explain `-100`

Why do labels use `-100` on padded positions?


<details>
<summary>Show Answer</summary>

PyTorch's cross-entropy loss ignores labels with value `-100`. Masking padding prevents the model from learning padding tokens, and masking the prompt focuses the update on the desired answer rather than on copying the instruction.

</details>


## Attach LoRA and Train

Only LoRA adapter parameters are optimized. The loop shows the forward pass, loss, backward pass, optimizer step, and gradient reset explicitly.

This demonstration uses rank 8 adapters on attention query and value projections, a relatively high learning rate, and six passes over twelve examples. Those choices intentionally encourage visible learning on a tiny dataset; a real project would tune them on a larger training set and stop according to validation behavior.


In [ ]:
base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, revision=MODEL_REVISION)
base_model.config.pad_token_id = tokenizer.eos_token_id
base_model.config.use_cache = False
base_model.loss_type = "ForCausalLM"

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    bias="none",
)
model = get_peft_model(base_model, lora_config)

In [ ]:
def trainable_parameter_names(model_to_inspect: torch.nn.Module) -> list[str]:
    # Adapter training should expose only LoRA parameters as trainable.
    return [
        name
        for name, parameter in model_to_inspect.named_parameters()
        if parameter.requires_grad
    ]


trainable_parameter_names(model)[:6]

### Exercise: Verify Adapter Training

Check that every trainable parameter name contains `lora`.


In [ ]:
# @TODO: Verify trainable parameter names refer to LoRA parameters.


<details>
<summary>Show Answer</summary>

```python
all("lora" in name for name in trainable_parameter_names(model))
```

</details>


In [ ]:
LEARNING_RATE = 2e-3
optimizer = torch.optim.AdamW(
    [parameter for parameter in model.parameters() if parameter.requires_grad],
    lr=LEARNING_RATE,
)

EPOCHS = 6
model.to(DEVICE)
model.train()
loss_history: list[float] = []
epoch_losses: list[float] = []

for epoch in range(EPOCHS):
    current_epoch_losses: list[float] = []
    for batch in train_loader:
        # Keep every tensor on the same device as the model.
        batch = {key: value.to(DEVICE) for key, value in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss
        if not torch.isfinite(loss):
            raise RuntimeError("Non-finite training loss; inspect the token labels.")
        # Backpropagation reaches only parameters marked as trainable by PEFT.
        loss.backward()
        optimizer.step()
        # Clear gradients before the next mini-batch.
        optimizer.zero_grad(set_to_none=True)
        loss_value = float(loss.detach().cpu())
        loss_history.append(loss_value)
        current_epoch_losses.append(loss_value)

    mean_epoch_loss = sum(current_epoch_losses) / len(current_epoch_losses)
    epoch_losses.append(round(mean_epoch_loss, 3))
    print(f"Epoch {epoch + 1}/{EPOCHS}: mean loss = {mean_epoch_loss:.3f}")

model.eval()
epoch_losses

The loss should generally fall because this deliberately small run revisits the same examples. That proves the adapter can fit the training signal, not that it will generalize. The held-out cases in notebook 04 are therefore more important than the training loss itself.


## Save the Adapter

The output folder is ignored by git because participants regenerate adapters locally. A saved adapter is small compared with a full model checkpoint.


In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
model.peft_config["default"].revision = MODEL_REVISION
# Only query/value adapters were trained; token embeddings are unchanged.
model.save_pretrained(str(OUTPUT_DIR), save_embedding_layers=False)
tokenizer.save_pretrained(str(OUTPUT_DIR))
training_summary = {
    "base_model": MODEL_NAME,
    "revision": MODEL_REVISION,
    "training_sha256": hashlib.sha256(training_bytes).hexdigest(),
    "prompt_format": "native-chat",
    "system_message": SYSTEM_MESSAGE,
    "seed": SEED,
    "epochs": EPOCHS,
    "learning_rate": LEARNING_RATE,
    "max_length": MAX_LENGTH,
    "epoch_losses": epoch_losses,
}
(OUTPUT_DIR / "training_summary.json").write_text(
    json.dumps(training_summary, indent=2), encoding="utf-8"
)
sorted(path.name for path in OUTPUT_DIR.iterdir())

### Exercise: Locate Adapter Weights

Which saved file contains the adapter tensor weights?


<details>
<summary>Show Answer</summary>

The adapter tensor weights are stored in `adapter_model.safetensors`. The configuration is stored in `adapter_config.json`.

</details>


## Smoke-Test Generation

The adapted model should now produce a readable support-style continuation for at least some familiar domains. The prompt below is not in the training data, so the output shows whether the adapter transfers beyond the examples it memorized. One output is only a smoke test; notebook 04 compares both models across every held-out case.


In [ ]:
def generate_reply(model_to_use, instruction: str) -> str:
    # Decode only the generated assistant turn, not the input conversation.
    prompt = build_inference_prompt(instruction)
    encoded = tokenizer(prompt, return_tensors="pt", add_special_tokens=False).to(
        DEVICE
    )
    generated = model_to_use.generate(
        **encoded,
        max_new_tokens=48,
        use_cache=True,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )
    new_tokens = generated[0, encoded["input_ids"].shape[1] :]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


generate_reply(
    model,
    "A customer asks whether their refund will go back to the card they paid with.",
)

## Further Reading

- [PEFT Quicktour](https://huggingface.co/docs/peft/quicktour)
- [Text Generation with LLMs](https://huggingface.co/docs/transformers/llm_tutorial)
- [Safetensors Documentation](https://huggingface.co/docs/safetensors/index)